<a href="https://colab.research.google.com/github/shymashbankhlyft-creator/Customer-task-NTI/blob/main/Wine_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ydata-profiling
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from ydata_profiling import ProfileReport
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    confusion_matrix
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42

In [ ]:
wine = load_wine()

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y_true = pd.Series(wine.target, name="true_label")  # only used for evaluation, NOT for training

print("Shape of the dataset:", X.shape)
print("Classes:", dict(zip(range(len(wine.target_names)), wine.target_names)))
X.head()

In [ ]:
profile = ProfileReport(X, title="Profiling Report")
profile

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

X_scaled.head()

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

plt.figure()
plt.bar(range(1, len(explained) + 1), explained, alpha=0.6, label="Individual explained variance")
plt.step(range(1, len(cumulative) + 1), cumulative, where="mid", color="red", label="Cumulative explained variance")
plt.axhline(0.90, color="green", linestyle="--", label="90% threshold")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA Explained Variance")
plt.legend()
plt.show()

n_components_90 = np.argmax(cumulative >= 0.90) + 1
print(f"Number of components needed to explain >=90% variance: {n_components_90}")

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
X_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])

X_pca.head()
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Singular values:", pca.singular_values_)

plt.figure(figsize=(8, 6))
sns.scatterplot(x="PC1", y="PC2", hue=y_true, data=X_pca, palette="viridis")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection")
plt.show()


In [ ]:
inertias = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_pca, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].set_title("Elbow Method")

axes[1].plot(list(k_range), silhouette_scores, marker="o", color="orange")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score vs k")

plt.tight_layout()
plt.show()

best_k = list(k_range)[int(np.argmax(silhouette_scores))]
print(f"Best k according to silhouette score: {best_k}")

In [ ]:
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_plot = pca2.fit_transform(X_scaled)

km = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
labels = km.fit_predict(X_plot)

plt.figure(figsize=(8, 6))
plt.scatter(X_plot[:, 0], X_plot[:, 1], c=labels, cmap='viridis', s=50)
plt.scatter(km.cluster_centers_[:, 0],
            km.cluster_centers_[:, 1],
            c='red', marker='X', s=200, label='Centroids')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means on 2D PCA Projection')
plt.legend()
plt.show()